### Project - Airline AI Assistant

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)

GEMINI_BASE_URL = os.getenv('GEMINI_BASE_URL')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

if GEMINI_API_KEY:
    print(f'GEMINI API Key found and starts with {GEMINI_API_KEY[0:3]}')
else:
    print(f'GEMINI API not found')

MODEL = 'gemini-3.5-flash-lite'

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)

GEMINI API Key found and starts with AQ.


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system','content':system_message}] + history + [{'role':'user', 'content':message}]
    try:
        stream = gemini.chat.completions.create(
            model=MODEL,
            messages=messages,
            stream=True
        )

        result = ""
        for chunk in stream:
            result += chunk.choices[0].delta.content or ""
            yield result
    except Exception as e:
        print(f'Exception: {e}')

In [5]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


#### Providing a separate function of ticket prices as a Tool

In [34]:
# Function as a Tool

tickets_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f'Tool called for city {destination_city}')
    price = tickets_prices.get(destination_city.lower(),'Unknown ticket price')
    return f'The price of the ticket to {destination_city} is {price}'

In [7]:
get_ticket_price('London')

Tool called for city London


'The price of the ticket to London is $799'

In [8]:
# Function Calling API expects this format of contract

my_tool_schema = {
    "name" : "get_ticket_price",
    "description" : "Get the ticket price of a return ticket to the destination city",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "destination_city" : {
                "type" : "string",
                "description" : "The city that the customer wants to travel to",
            },
        },
        "required" : ["destination_city"],
        "additional_parameters" : False
    }
}

# Particular dictionary structure that's required to call the function
# my_tool_schema tells the model how to call get_ticket_price

In [9]:
# lost of tools

tools = [{'type':'function', 'function':my_tool_schema}]

# the above line indicates the type of the tool, which is a function and all the attributes and metadata about that tool or function

In [10]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the ticket price of a return ticket to the destination city',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additional_parameters': False}}}]

#### Getting the Model to use Tools

In [11]:
# Informing the LLM about the available tools

def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history + [{'role':'user', 'content':message}]
    response = gemini.chat.completions.create(
        model = MODEL,
        messages=messages,
        tools=tools                                                     # information about the tools has been passed here
    )
    print(f'response : {response}')
    print('-'*20)

    if response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        print(f'Calling handle_tool_call function with message : {message}')
        print('-'*20)
        response = handle_tool_call(message)
        print(f'handle_tool_call function returned the response : {response}')
        messages.append(message)
        messages.append(response)
        response = gemini.chat.completions.create(
            model = MODEL,
            messages=messages
        )
    return response.choices[0].message.content



In [12]:
# function to handle the tool call

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == 'get_ticket_price':
        arguments = json.loads(tool_call.function.arguments)
        print(f'Arguments : {arguments}')
        print('-'*20)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            'role' : 'tool',
            'content' : price_details,
            'tool_call_id' : tool_call.id
        }
    return response

In [ ]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


response : ChatCompletion(id='KQV0ao-VMtqejuMPm8y3mAU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello, how can I help you with your travel plans today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EjQKMgERTTIP6Nq3JLKhgH4BT5BATWTi4IHPoZrCMqbrsbKdWLn0cqhoFvVoqwoI+/Bp0fbL'}}))], created=1785988394, model='gemini-3.5-flash-lite', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=13, prompt_tokens=117, total_tokens=130, completion_tokens_details=None, prompt_tokens_details=None))
--------------------
response : ChatCompletion(id='WQV0aviTIImxjuMP44aegAw', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_ca

### To make the code to handle multiple tool calls in one response

- In the previous code, if we pass 

    - I am looking to travel to the cheapest destination among berlin and london. Give me which one is cheaper among them.

    - Output is: "I do not have the current ticket price for London to compare, so I am unable to tell you which destination is cheaper."

    - This is because, our previous code can't handle multiple tool calls in one response.

    - Since, the berlin came first it stopped there and there is no tool handling for london in order to give proper response

In [ ]:
def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history + [{'role':'user', 'content':message}]
    response = gemini.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = tools
    )

    if response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)                                      # we are using extend here as handle_tool_calls() returns the list of responses for multiple cities
        response = gemini.chat.completions.create(
            model = MODEL,
            messages = messages                                       
        )
    return response.choices[0].message.content


In [ ]:
# to handle multiple tool calls in a response

def handle_tool_calls(message):
    responses = []                                                      # multiple tools calls contains multiple responses, so we are using list of responses to return
    for tool_call in message.tool_calls:                                # to handle mutiple tool calls
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                'role':'tool',
                'content':price_details,
                'tool_call_id':tool_call.id
            })
    return responses


''' 
This code perfectly works for:  (handling multiple tool calls in one response)

    tell me the ticket prices for berlin and london

But fails for:                  (handling multiple tool calls in 1 after the other. Sequential calls)

    tell me the ticket price for london and only if it is under 1000, then tell me the ticket price for paris as well

'''



In [43]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Tool called for city Berlin
Tool called for city Tokyo
Tool called for city London


Traceback (most recent call last):
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 870, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 408, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2316, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dorababulalam/GitHub/gen-ai/llm_engineering_practice/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1681, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^

### To handle multiple tools calls one after the other

- Tell me the ticket price for london and only if it is under 1000, then tell me the ticket price for paris as well

In [ ]:
def chatbot(message, history):
    history = [{'role':h['role'], 'content':h['content']} for h in history]
    messages = [{'role':'system', 'content':system_message}] + history + [{'role':'user', 'content':message}]
    response = gemini.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = tools
    )

    while response.choices[0].finish_reason == 'tool_calls':        # to check if there is a need for tool calling in sequential operations
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(
            model = MODEL,
            messages = messages,
            tools = tools
        )
    return response.choices[0].message.content

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                'role' : 'tool',
                'content' : price_details,
                'tool_call_id' : tool_call.id
            })
    return responses

''' 
This code works for both the cases:

1. give me the ticket prices for both berlin and tokyo  - [multiple tools calls in 1 response]

2. give me the ticket price for london and only if the price is below, then give me the price for paris as well -   [multiple tool calls one after the other]

'''

In [53]:
gr.ChatInterface(fn=chatbot).launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Tool called for city Berlin
Tool called for city Tokyo
Tool called for city london
Tool called for city paris
